In [1]:
import pandas as pd 
import numpy as np 
from matplotlib import pyplot as plt 
import seaborn as sns 

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [2]:
dfx = pd.read_csv('exp1_search_space.csv')
# limit to budget = 300
dfx = dfx[dfx.time<=300]
dfx['num_replicates'] = dfx.groupby(['method', 'p1', 'p2']).cumcount() + 1
# include initial design 

dfx_init = dfx[dfx.iter==1].drop_duplicates(subset=['p1', 'p2', 'num_replicates'])
dfx_init['method'] = "Initial Design"

dfx = pd.concat([dfx, dfx_init])
dfx.head()

,p1,p2,rep,rmse,method,time,iter,num_replicates
0,0.841195,0.835311,1,43.450438,fCRN,1,1,1
1,0.841195,0.835311,2,41.686477,fCRN,2,1,2
2,0.841195,0.835311,3,59.818951,fCRN,3,1,3
3,0.841195,0.835311,4,20.534684,fCRN,4,1,4
4,0.841195,0.835311,5,42.003772,fCRN,5,1,5


In [9]:
all_methods = [['Initial Design', 'fHet', 'aHet'],
               ['fgCRN', 'fCRN', 'aCRN']]
n=6
fig = make_subplots(
    rows=2, cols=3,
    specs=[[{'type': 'surface'}]*3, [{'type':'surface'}]*3],
    subplot_titles=all_methods[0]+all_methods[1],
    horizontal_spacing=0,   # shrink gutters 
    vertical_spacing=0
)

camera = dict(
    eye=dict(x=1.7,   # out to the right
             y=-1.2,  # pulled forward
             z=0.9),  # lifted up
    up=dict(x=0, y=0, z=1)  # ensure “up” is along the z–axis
)

# Prepare the plane at x = 20, over p1,p2 ∈ [0,1]
x0 = 20
grid_n = 10
p1_lin = np.linspace(0, 1, grid_n)
p2_lin = np.linspace(0, 1, grid_n)
P1, P2 = np.meshgrid(p1_lin, p2_lin)
X = np.full_like(P1, x0)

rep_val = 'num_replicates'
ground_truth = {'p1':0.45, 'p2':0.35, 'rep':-1}

for i, methods in enumerate(all_methods, start=1):
    for j, method in enumerate(methods, start=1):
        dfm = dfx[(dfx['method']==method)]
        if method != 'Initial Design':
            dfm = dfm[dfm.iter>1]

        # build a color list: blue if <20, red otherwise
        colors = ['tomato' if rr <= 20 else 'tomato' for rr in dfm['num_replicates']]
        
        fig.add_trace(
            go.Scatter3d(
                x = dfm[rep_val],
                y = dfm['p1'],
                z = dfm['p2'],
                mode = 'markers',
                marker = dict(
                    size = 3,
                    color = colors,
                    opacity=1
                ),
                showlegend = False
            ),
            row=i, col=j
        )
        # fig.add_trace(
        #     go.Scatter3d(
        #         x=[ground_truth['rep']],
        #         y=[ground_truth['p1']],
        #         z=[ground_truth['p2']],
        #         mode='markers',
        #         marker = dict(
        #             size=5,
        #             color='black'
        #         ),
        #         showlegend=False
        #     ),
        #     row=i, col=j
        # )
        fig.add_trace(
            go.Surface(
                x = X, y = P1, z = P2,
                # make all values in this matrix the same (e.g. zeros)
                surfacecolor = np.zeros_like(P1),
                # map “0” → light-green; “1” → light-green (so it’s uniform)
                colorscale = [[0,'cornflowerblue'], [1, 'cornflowerblue']],
                opacity    = 0.5,
                showscale  = False,
                showlegend = False
            ),
            row=i, col=j
        )
        
        fig.update_scenes(
            dict(
                # overall scene background
                bgcolor='white',
                aspectmode='cube',
                # x-axis (num_replicates), descending
                xaxis=dict(
                    range=[dfx[rep_val].max(), -1],
                    title='Replicates',
                    showgrid=False,    # no grid‐planes
                    zeroline=False,    # no zero‐plane
                    showline=True,     # draw the axis line
                    mirror=True,       # mirror that line on the opposite side of the box
                    linewidth=2,       # thickness
                    linecolor='black',  # color of the box edges
                    showticklabels=False,
                    backgroundcolor='white'

                ),
                # y-axis (p1)
                yaxis=dict(
                    title='β',
                    range=[0, 1],
                    showgrid=False,
                    zeroline=False,
                    showline=True,
                    mirror=True,
                    linewidth=2,
                    linecolor='black',
                    showticklabels=False,
                    backgroundcolor='white'
            
                ),
                # z-axis (p2)
                zaxis=dict(
                    range=[0, 1],
                    title='γ',
                    showgrid=False,
                    zeroline=False,
                    showline=True,
                    mirror=True,
                    linewidth=2,
                    linecolor='black',
                    showticklabels=False,
                    backgroundcolor='white'
                ),
                camera = camera
            ),
            row=i, col=j
        )

annos = []
for i, methods in enumerate(all_methods):
    for j, method in enumerate(methods):
        x = (j + 0.5)/3 
        if i == 0:
            y = .925
        elif i == 1:
            y = .425
        annos.append(dict(
            text=method,
            x=x, y=y,          # just above the top of the plot
            xref='paper', yref='paper',
            showarrow=False,
            font=dict(size=18),
        ))
    
fig.update_layout(
    width = 150 * n,              # ← bump per‐panel width
    height = 600,                 # ← bump height for more vertical space
    margin = dict(l=0, r=0, t=50, b=0),  # trim outer margins
    paper_bgcolor='white',
    annotations=annos,
)

    
fig.show()

In [8]:
all_methods = [['Initial Design', 'fHet', 'aHet'],
               ['fgCRN', 'fCRN', 'aCRN']]
n=6
fig = make_subplots(
    rows=2, cols=3,
    specs=[[{'type': 'surface'}]*3, [{'type':'surface'}]*3],
    subplot_titles=all_methods[0]+all_methods[1],
    horizontal_spacing=0,   # shrink gutters 
    vertical_spacing=0
)

camera = dict(
    eye=dict(x=1.7,   # out to the right
             y=-1.2,  # pulled forward
             z=0.9),  # lifted up
    up=dict(x=0, y=0, z=1)  # ensure “up” is along the z–axis
)

# Prepare the plane at x = 20, over p1,p2 ∈ [0,1]
x0 = 20
grid_n = 10
p1_lin = np.linspace(0, 1, grid_n)
p2_lin = np.linspace(0, 1, grid_n)
P1, P2 = np.meshgrid(p1_lin, p2_lin)
X = np.full_like(P1, x0)

rep_val = 'rep'
ground_truth = {'p1':0.45, 'p2':0.35, 'rep':50}

for i, methods in enumerate(all_methods, start=1):
    for j, method in enumerate(methods, start=1):
        dfm = dfx[(dfx['method']==method)]
        if method != 'Initial Design':
            dfm = dfm[dfm.iter>1]

        # build a color list: blue if <20, red otherwise
        colors = ['tomato' if rr <= 20 else 'tomato' for rr in dfm['num_replicates']]
        
        fig.add_trace(
            go.Scatter3d(
                x = dfm[rep_val],
                y = dfm['p1'],
                z = dfm['p2'],
                mode = 'markers',
                marker = dict(
                    size = 3,
                    color = colors,
                    opacity=1
                ),
                showlegend = False
            ),
            row=i, col=j
        )
        fig.add_trace(
            go.Scatter3d(
                x=[ground_truth['rep']],
                y=[ground_truth['p1']],
                z=[ground_truth['p2']],
                mode='markers',
                marker = dict(
                    size=6,
                    color='black'
                ),
                showlegend=False
            ),
            row=i, col=j
        )
        fig.add_trace(
            go.Surface(
                x = X, y = P1, z = P2,
                # make all values in this matrix the same (e.g. zeros)
                surfacecolor = np.zeros_like(P1),
                # map “0” → light-green; “1” → light-green (so it’s uniform)
                colorscale = [[0,'cornflowerblue'], [1, 'cornflowerblue']],
                opacity    = 0.5,
                showscale  = False,
                showlegend = False
            ),
            row=i, col=j
        )
        
        fig.update_scenes(
            dict(
                # overall scene background
                bgcolor='white',
                aspectmode='cube',
                # x-axis (num_replicates), descending
                xaxis=dict(
                    range=[dfx[rep_val].max(), -1],
                    title='r',
                    showgrid=False,    # no grid‐planes
                    zeroline=False,    # no zero‐plane
                    showline=True,     # draw the axis line
                    mirror=True,       # mirror that line on the opposite side of the box
                    linewidth=2,       # thickness
                    linecolor='black',  # color of the box edges
                    showticklabels=False,
                    backgroundcolor='white'

                ),
                # y-axis (p1)
                yaxis=dict(
                    title='β',
                    range=[0, 1],
                    showgrid=False,
                    zeroline=False,
                    showline=True,
                    mirror=True,
                    linewidth=2,
                    linecolor='black',
                    showticklabels=False,
                    backgroundcolor='white'
            
                ),
                # z-axis (p2)
                zaxis=dict(
                    range=[0, 1],
                    title='γ',
                    showgrid=False,
                    zeroline=False,
                    showline=True,
                    mirror=True,
                    linewidth=2,
                    linecolor='black',
                    showticklabels=False,
                    backgroundcolor='white'
                ),
                camera = camera
            ),
            row=i, col=j
        )

annos = []
for i, methods in enumerate(all_methods):
    for j, method in enumerate(methods):
        x = (j + 0.5)/3 
        if i == 0:
            y = .925
        elif i == 1:
            y = .425
        annos.append(dict(
            text=method,
            x=x, y=y,          # just above the top of the plot
            xref='paper', yref='paper',
            showarrow=False,
            font=dict(size=18),
        ))
    
fig.update_layout(
    width = 150 * n,              # ← bump per‐panel width
    height = 600,                 # ← bump height for more vertical space
    margin = dict(l=0, r=0, t=50, b=0),  # trim outer margins
    paper_bgcolor='white',
    annotations=annos,
)

    
fig.show()